In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

定义模型

这里需要识别图片，所以需要定义多模态模型，不能使用Deepseek

In [4]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model = "qwen3.7-plus",   # 多模态模型，支持图片，文本，音频，视频
    model_provider = "openai",
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
    api_key = os.getenv("DASHSCOPE_API_KEY"),
)

定义工具

In [5]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results = 5,
    topic = "general",
)


添加记忆管理

采用sqlite

In [8]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 连接sqlite
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread = False)

# 初始化checkpointer
checkpointer = SqliteSaver(connection)

# 自动建表
checkpointer.setup()

定义知识体

In [11]:
from langchain.agents import create_agent

system_prompt ="""
下是一名私人厨师。收到用户提供的食材照片或清单后,请按以下流程操作:
1.识别和评估食材:若用户提供照片,首先辨识所有可见食材。基于食材的外观状态,评估其新鲜度与可用量,整理出一份“当前可用食材清单”。
2.智能食谱检索:优先调用 web_search 工具,以“可用食材清单”为核心关键词,查找可行菜谱。
3.多维度评估与排序:从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分,并根据得分排序,制作简单且营养丰富的排名靠前。
4.结构化方案输出:把排序后的食谱整理为一份结构清晰的建议报告,要包含食谱信息、得分、推荐理由、食谱的参考图片,帮助用户快速做出决策。

请严格按照流程,优先调用 web_search 工具搜索食谱,搜索不到的情况下才能自己发挥。

"""

agent = create_agent(
    model = model,
    tools = [web_search],
    system_prompt = system_prompt,
    checkpointer = checkpointer,
)

测试

In [12]:
from langchain.messages import HumanMessage

multinomial_message = HumanMessage(
    [
        {"type": "text", "text": "帮我看看能做什么。"},
        {"type":"image", "url": "https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg"}
    ]
)

config = {"configurable": {"thread_id": "1"}}

In [13]:
response = agent.invoke({"messages": [multinomial_message]}, config)

In [15]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么。'}, {'type': 'image', 'url': 'https://aisearch.cdn.bcebos.com/pic_create/2026-04-10/10/74d52055e4947f8c.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_70b8132b581748f9a39bd0b2)
 Call ID: call_70b8132b581748f9a39bd0b2
  Args:
    query: 西兰花番茄青椒茄子白菜家常菜谱做法
    search_depth: advanced
    include_images: true
  tavily_search (call_ef406aefa309400da9d4f0e7)
 Call ID: call_ef406aefa309400da9d4f0e7
  Args:
    query: 腊肠笋子虾豆角家常菜谱做法
    search_depth: advanced
    include_images: true
  tavily_search (call_6e3a26fb35bd4f3db2c41362)
 Call ID: call_6e3a26fb35bd4f3db2c41362
  Args:
    query: 番茄炒蛋青椒炒茄子白菜豆腐简单营养菜谱
    search_depth: advanced
    include_images: true
================================= Tool Message =================================
Name: tavily_search

{"query": "西兰花番茄青椒茄子白菜家常菜谱做法", "follow_up_quest

In [17]:
response = agent.invoke({"messages": [HumanMessage(content = "我喜欢第二道菜，能说的更详细吗？")]}, config)

In [18]:
print(response['messages'][-1].pretty_print())

================================== Ai Message ==================================

太棒了！您的眼光真好！**「茄子番茄青椒」** 这道菜被称为"米饭杀手"，不仅因为做法极简，更因为茄子软糯、番茄酸甜、青椒清脆，三种口感在嘴里碰撞，超级下饭！

下面我为您把这道菜拆解成**保姆级详细教程**，保证您一次成功！

---

### 🍆 菜名：茄子番茄青椒（5分钟快手下饭菜）

#### 🛒 一、 食材清单（基于您的冰箱）
| 食材 | 用量 | 处理建议 |
|------|------|----------|
| 🍆 茄子 | 1根（中等大小） | 洗净，去蒂，切成约3-4厘米的长段或滚刀块 |
| 🍅 番茄 | 1-2个 | 洗净，去蒂，切成小块（越小越容易出汁） |
| 🫑 青椒 | 1-2个 | 洗净，去籽，切成和茄子差不多大小的段 |
| 🧄 蒜末 | 3-4瓣 | 切成细末（蒜香是这道菜的灵魂） |

#### 🥣 二、 灵魂料汁（提前调好，炒菜不慌）
找一个小碗，按以下比例混合均匀：
- **生抽**：2勺（提鲜增咸）
- **蚝油**：1勺（增加浓郁口感）
- **白糖**：半勺（中和番茄的酸，提鲜）
- **盐**：少许（因为生抽和蚝油已有咸味，盐一定要少放）
- **香油/芝麻油**：半勺（出锅前增香）
- *(可选) 淀粉*：半勺（如果喜欢汤汁浓稠挂在菜上，可以加一点水淀粉）

---

### 👨‍🍳 三、 详细制作步骤

#### 方案 A：极简微波炉版（最省油、最快手，适合懒人）
1. **微波加热**：将切好的茄子段和青椒段放入可微波的碗中，**不加水**，直接放入微波炉高火加热 **4-5分钟**。
2. **剪碎混合**：取出碗（小心烫），用厨房剪刀将变软的茄子和青椒剪成小块。
3. **加入番茄**：把切好的番茄块和蒜末放进去，再微波 **2分钟**，让番茄出汁。
4. **拌匀出锅**：取出后，倒入提前调好的"灵魂料汁"，搅拌均匀即可开吃！

#### 方案 B：传统少油炒制版（锅气更足，更香）
1. **处理茄子（防吸油秘诀）**：茄子切块后，撒少许盐拌匀，静置5分钟杀出水分，然后**用手挤干水分**。这一步能让茄子不吸油且更容易熟。
2. **炒番